# Movies Recommendation System - Generated By Zain Khan

### Load the data 

In [6]:

import pandas as pd
import numpy as np

# Load your downloaded CSV files 
# (Update the file names/paths if yours are named differently)
try:
    movies_df = pd.read_csv('D:/tmdb_5000_movies.csv')
    credits_df = pd.read_csv('D:/tmdb_5000_credits.csv')
    print(" Data loaded successfully!")
    print(f"Movies Shape: {movies_df.shape} | Credits Shape: {credits_df.shape}")
except FileNotFoundError as e:
    print(f"❌ Error: Could not find the files. Check your file paths. Details: {e}")

# Inspect the columns available to us
print("\nMovies Columns:", movies_df.columns.tolist())
print("Credits Columns:", credits_df.columns.tolist())

# Preview the first 2 rows of the movie titles
print("\nFirst 2 Movie Titles:")
print(movies_df['title'].head(2))


 Data loaded successfully!
Movies Shape: (4803, 20) | Credits Shape: (4803, 4)

Movies Columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']
Credits Columns: ['movie_id', 'title', 'cast', 'crew']

First 2 Movie Titles:
0                                      Avatar
1    Pirates of the Caribbean: At World's End
Name: title, dtype: str


In [7]:
movies_df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [8]:
movies_df.isnull().sum()

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
dtype: int64

# MERGE DATASETS & SELECT FEATURES

In [9]:
# 1. Rename 'id' column in movies_df to 'movie_id' if needed to match credits_df
if 'id' in movies_df.columns:
    movies_df = movies_df.rename(columns={'id': 'movie_id'})

# 2. Merge the datasets on 'movie_id'
# We merge on movie_id to ensure a perfect match, and drop duplicate title columns
if 'movie_id' in movies_df.columns and 'movie_id' in credits_df.columns:
    # Drop 'title' from credits to avoid duplicate columns after merge
    if 'title' in credits_df.columns:
        credits_df = credits_df.drop(columns=['title'])
    
    df = movies_df.merge(credits_df, on='movie_id')
    print(" Merging complete!")
else:
    # Fallback to merging on title if id isn't uniform
    df = movies_df.merge(credits_df, on='title')
    print(" Merged on 'title' column successfully.")

# 3. Keep only the core columns required for a content-based recommendation model
# These columns contain the vital textual "DNA" of the movie
required_columns = ['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']
df = df[required_columns]

# 4. Check for and handle missing values
print("\nMissing values per column before cleaning:")
print(df.isnull().sum())

# Drop rows where 'overview' or 'title' is missing (vital identifier fields)
df = df.dropna(subset=['title', 'overview'])

# Reset index to maintain perfect sequential numbering
df = df.reset_index(drop=True)

print(f"\n Cleaned Dataset Shape: {df.shape}")
print("\nPreview of the combined data dataframe:")
print(df[['title', 'genres']].head(3))


 Merging complete!

Missing values per column before cleaning:
movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

 Cleaned Dataset Shape: (4800, 7)

Preview of the combined data dataframe:
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   

                                              genres  
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...  
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...  
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...  


# PARSING JSON STRINGS TO CLEAN TEXT

In [10]:
import ast

# 1. Yeh function ajeeb text mein se sirf 'name' (Action, Sci-Fi) ko bahar nikaalega
def convert_features(text):
    L = []
    # ast.literal_eval is ajeeb string ko ek real Python list mein badalta hai
    for i in ast.literal_eval(text):
        L.append(i['name']) 
    return L

# Genres aur Keywords columns par is function ko apply karein
df['genres'] = df['genres'].apply(convert_features)
df['keywords'] = df['keywords'].apply(convert_features)


# 2. Cast column mein 50 actors ho sakte hain, humein sirf top 3 main actors chahiye
def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3: # Sirf pehle 3 main actors ki list banao
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

df['cast'] = df['cast'].apply(convert_cast)


# 3. Crew column mein poori team hoti hai (spotboy, makeup, camera). Humein sirf 'Director' chahiye.
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director': # Agar job Director hai, to naam save karlo
            L.append(i['name'])
            break # Director milte hi mazeed dhoondna band karo
    return L

df['crew'] = df['crew'].apply(fetch_director)

print(" Step 3 Completed! Data bilkul saaf ho chuka hai.")

# Dekhte hain hamara data ab kaisa dikh raha hai
print("\n--- Safai ke baad ka Preview ---")
print(df[['title', 'genres', 'keywords', 'cast', 'crew']].head(3))


 Step 3 Completed! Data bilkul saaf ho chuka hai.

--- Safai ke baad ka Preview ---
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   

                                          genres  \
0  [Action, Adventure, Fantasy, Science Fiction]   
1                   [Adventure, Fantasy, Action]   
2                     [Action, Adventure, Crime]   

                                            keywords  \
0  [culture clash, future, space war, space colon...   
1  [ocean, drug abuse, exotic island, east india ...   
2  [spy, based on novel, secret agent, sequel, mi...   

                                               cast              crew  
0  [Sam Worthington, Zoe Saldana, Sigourney Weaver]   [James Cameron]  
1     [Johnny Depp, Orlando Bloom, Keira Knightley]  [Gore Verbinski]  
2      [Daniel Craig, Christoph Waltz, Léa Seydoux]      [Sam Mendes]  


# REMOVE SPACES & CREATE TAG SOUP

In [11]:
# 1. Ek helper function jo list ke har word se spaces khatam karega
# Jaise: ['Action', 'Sci Fi'] -> ['Action', 'SciFi']
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

# Is function ko saare columns par apply karein
df['genres'] = df['genres'].apply(collapse)
df['keywords'] = df['keywords'].apply(collapse)
df['cast'] = df['cast'].apply(collapse)
df['crew'] = df['crew'].apply(collapse)

# 2. Overview column abhi ek lamba sentence (string) hai.
# Isko bhi baaki columns ki tarah alag-alag words ki list mein badal dete hain.
df['overview'] = df['overview'].apply(lambda x: x.split())

# 3. Create the "Tag Soup" - Saare columns ke words ko ek sath jama karein
df['tags'] = df['overview'] + df['genres'] + df['keywords'] + df['cast'] + df['crew']

# 4. Ab hamara kaam baki columns se khatam ho gaya. Ek naya clean dataframe banate hain.
new_df = df[['movie_id', 'title', 'tags']]

# 5. List of words ko wapas ek normal paragraph (string) mein convert karte hain
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

# 6. Saare words ko lower-case (chhoti ABC) mein convert karte hain taake computer confuse na ho
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

print(" Step 4 Completed! Tag Soup tayyar hai.")

# Dekhte hain hamara final data kaisa dikh raha hai
print("\n--- Final 'tags' ka Pehla Paragraph ---")
print(new_df['tags'].iloc[0][:300] + "...") # Pehle 300 characters


 Step 4 Completed! Tag Soup tayyar hai.

--- Final 'tags' ka Pehla Paragraph ---
in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance spac...


# TEXT VECTORIZATION (BAG OF WORDS)

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Vectorizer ka object banayein
# max_features=5000 matlab hum sirf top 5000 zaroori words nikalenge
# stop_words='english' matlab hum 'is', 'the', 'and', 'a' jaise faltu words ko hata denge
cv = CountVectorizer(max_features=5000, stop_words='english')

# 2. Apne tags column ko numbers (vectors) mein convert karein
vectors = cv.fit_transform(new_df['tags']).toarray()

print(" Step 5 Completed! Text successfully numbers mein badal gaya hai.")
print(f"Vectors ki Shakal (Shape): {vectors.shape}") 
# Output aayega (4806, 5000) -> Yani 4806 movies aur har movie ke 5000 numbers.

# Agar aap dekhna chahein ki computer ne kaunse 5000 words nikale hain:
# print(cv.get_feature_names_out()[:100]) # Pehle 100 words dekhne ke liye


 Step 5 Completed! Text successfully numbers mein badal gaya hai.
Vectors ki Shakal (Shape): (4800, 5000)


# COSINE SIMILARITY & RECOMMENDATION

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Saare vectors ke beech mein Cosine Similarity nikaalein
# Yeh ek (4806 x 4806) ka matrix banayega jo har movie ki har doosri movie se similarity batayega
similarity = cosine_similarity(vectors)

# 2. Asal Recommendation Function banayein
def recommend(movie):
    # User jo movie input karega, pehle dataset mein uski index position dhoondo
    try:
        movie_index = new_df[new_df['title'].str.lower() == movie.lower()].index[0]
    except IndexError:
        print(f"❌ Error: '{movie}' hamare dataset mein nahi mili. Spelling check karein.")
        return

    # Us movie ke similarity scores nikaalo aur index ke sath pair (enumerate) karo
    distances = similarity[movie_index]
    
    # Scores ko descending order (zyada se kam) mein sort karo
    # lambda x: x[1] ka matlab hai hum similarity score ki buniyaad par sort kar rahe hain
    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    # Top 5 similar movies ke naam print karo
    print(f"\n🎬 If you liked '{movie}', you should definitely watch:")
    for i in movie_list:
        print(f"👉 {new_df.iloc[i[0]].title}")

# ==========================================
# TEST YOUR MODEL HERE! 🎉
# ==========================================
recommend('Avatar')
# recommend('The Dark Knight Rises')
# recommend('Iron Man')



🎬 If you liked 'Avatar', you should definitely watch:
👉 Titan A.E.
👉 Small Soldiers
👉 Ender's Game
👉 Independence Day
👉 Aliens vs Predator: Requiem


In [ ]:
import pickle

# Naya dataframe aur similarity matrix ko files mein save kar rahe hain
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print("Files saved successfully! Ab aap notebook band kar sakte hain.")
